# LangChain LLMOps Course Summary

This notebook is a guide to the 11 lesson notebooks in this folder. The sequence moves from direct model calls to prompts and parsers, document retrieval, LCEL composition, conversational memory, tool-using agents, and LangSmith tracing.

The examples use OpenAI models in most lessons, with optional Groq, Tavily, Chroma, FAISS, Wikipedia, web, text, CSV, HTML, and PDF integrations.

## Learning Path

1. **Model interaction:** invoke and stream completion and chat models.
2. **Prompt engineering:** templates, few-shot examples, chains, and structured output.
3. **RAG foundations:** load documents, split text, create embeddings, index vectors, and retrieve context.
4. **LCEL:** compose runnables with `|`, pass data through chains, run work in parallel, stream, and batch.
5. **Stateful applications:** preserve conversation history and limit the memory window.
6. **Agents and observability:** call search tools, retain graph state, and trace chains with LangSmith.

## Notebook-by-Notebook Summary

### 01_talk_with_llm.ipynb - Direct model calls
Introduces `OpenAI` completion models and `ChatOpenAI` chat models. Covers `invoke`, streaming, temperature, message roles, response content and metadata, and an alternative `ChatGroq` provider.

### 02_talk_with_llm_part2.ipynb - Prompts and output formats
Builds `PromptTemplate` and `ChatPromptTemplate` objects, demonstrates few-shot prompting and legacy chains, then uses `SimpleJsonOutputParser` and `JsonOutputParser` with a Pydantic schema for reliable structured results.

### 03_Data_loader_rag_demo.ipynb - Loading external data
Loads TXT, CSV, HTML, PDF, and Wikipedia content with LangChain document loaders. The loaded documents are passed into a chat prompt so the model can answer using supplied context.

### 04_RAG_components.ipynb - RAG building blocks
Explains character and recursive text splitting, chunk size and overlap, embeddings, vector stores, similarity search, and retrievers using Chroma and FAISS.

### 05_Basic_RAG_Implementation.ipynb - A complete RAG chain
Loads the State of the Union text, splits it, embeds it into FAISS, retrieves relevant passages, formats documents, and composes retriever, prompt, chat model, and string parser into an LCEL question-answering chain.

### 06_LCEL.ipynb - LangChain Expression Language
Introduces the pipe operator for composing prompts, models, and parsers. Compares the older `LLMChain` API with LCEL and demonstrates streaming and batching.

### 07_LCEL_builtin_runnables.ipynb - Reusable runnable primitives
Covers `RunnablePassthrough`, `RunnableLambda`, and `RunnableParallel`, including dictionary-shaped inputs and a parallel RAG-style composition.

### 08_LCEL_little_more.ipynb - Advanced composition
Demonstrates `.bind()` for model arguments, chains nested inside other chains, `itemgetter`, and a web-based RAG chain that uses LangSmith-related components.

### 09_Memory_langchain.ipynb - Conversation state
Uses `RunnableWithMessageHistory` and `InMemoryChatMessageHistory` for multi-turn conversations. Shows full buffer memory, a trimmed conversation window, session IDs, and direct message history operations.

### 11_langgraph_demo.ipynb - Tool-using agents
Creates a ReAct-style agent with a Tavily search tool. Demonstrates invocation, streaming, and `MemorySaver` checkpoints so different thread IDs can keep separate conversational state.

### 12_langsmith_demo.ipynb - Tracing and evaluation visibility
Enables LangSmith tracing through environment variables and sends a prompt-model-parser chain to LangSmith while answering from supplied context.

In [ ]:
lessons = {
    1: 'Direct LLM and chat model calls',
    2: 'Prompt templates and structured output',
    3: 'Document loaders',
    4: 'RAG components',
    5: 'Basic RAG implementation',
    6: 'LCEL fundamentals',
    7: 'Built-in LCEL runnables',
    8: 'Advanced LCEL composition',
    9: 'Conversation memory',
    11: 'LangGraph agents and checkpoints',
    12: 'LangSmith tracing',
}

for number, topic in lessons.items():
    print(f'{number:02d}: {topic}')

## Core Concepts

- **Model:** generates a completion or chat message.
- **Prompt:** turns application inputs and instructions into model input.
- **Output parser:** converts model output into a string, JSON object, or typed schema.
- **Document:** text plus metadata loaded from a source.
- **Chunk:** a smaller document segment used for indexing and retrieval.
- **Embedding:** a numeric vector representing semantic meaning.
- **Vector store:** indexes embeddings for similarity search.
- **Retriever:** returns relevant documents for a query.
- **Runnable:** a composable unit that accepts input and produces output.
- **Memory/checkpoint:** stores conversation or graph state across calls.
- **Trace:** records chain and model execution for debugging and evaluation.

## End-to-End RAG Architecture

```text
source files or web pages
        |
        v
document loader -> text splitter -> embeddings -> vector store
                                                   |
user question -> retriever ------------------------+
                         |
                         v
                 context + question
                         |
                         v
                 prompt -> chat model -> output parser -> answer
```

The key design choice is to ground the prompt with retrieved context and explicitly instruct the model to answer from that context. This reduces unsupported answers, but retrieval quality, chunking, prompt wording, and model behavior still need testing.

## Environment and Setup Notes

- Install the packages listed in `requirements.txt`.
- Load secrets from a local `.env` file; do not hard-code API keys in notebooks.
- Common variables include `OPENAI_API_KEY`, `LANGCHAIN_API_KEY`, `LANGCHAIN_TRACING_V2`, and `TAVILY_API_KEY`.
- Run notebooks from the repository root so paths such as `./data/state_of_the_union.txt` resolve correctly.
- Network-backed examples require internet access and may incur provider costs.
- Model names and LangChain imports can change over time, so check current package documentation when an example stops working.

## Common Pitfalls

- Missing or incorrectly loaded environment variables cause authentication failures.
- Relative data paths depend on the notebook's working directory.
- Large chunks can reduce retrieval precision; tiny chunks can lose context. Tune chunk size and overlap for the source.
- A retriever returning documents does not guarantee that the model will use them. Keep the prompt explicit and inspect retrieved context.
- Memory is keyed by session or thread ID. Reusing an ID shares state; changing it starts a separate conversation.
- Legacy APIs such as `LLMChain` appear for comparison, but newer examples favor LCEL and runnable history.
- Tracing should be disabled or carefully configured when prompts contain sensitive data.

## Suggested Next Steps

1. Replace hard-coded model names with configuration.
2. Add retrieval tests with known questions and expected source documents.
3. Compare chunk sizes, overlap, `k`, and vector stores on the same evaluation set.
4. Add citations or source metadata to RAG responses.
5. Persist chat history and LangGraph checkpoints beyond in-memory storage.
6. Use LangSmith traces to compare latency, token usage, retrieval quality, and answer correctness.
7. Move repeated notebook code into small Python modules once the experiments stabilize.

## 01_talk_with_llm.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]
openai_api_key

In [ ]:
# Completion model

from langchain_openai import OpenAI

llmModel = OpenAI()

In [ ]:
response  = llmModel.invoke("Tell me one fun fact about the Kennedy family.")

In [ ]:
for chunk in llmModel.stream(
    "Tell me one fun fact about the Kennedy family.i need detail response"
):
    print(chunk, end="", flush= True)

In [ ]:
creativeLlmModel = OpenAI(temperature=0.9)

In [ ]:
response  = creativeLlmModel.invoke("Tell me one fun fact about the Kennedy family.")

In [ ]:
response = llmModel.invoke(
    "Write a short 5 line poem about JFK"
)

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
messages = [
    ("system", "You are an historian expert in the Kennedy family."),
    ("human", "Tell me one curious thing about JFK."),
]
response = chatModel.invoke(messages)

In [ ]:
# Where you see this:

llm = ChatOpenAI(model="gpt-3.5-turbo")

# Write this instead:

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
messages = [
    SystemMessage(content="You are an historian expert on the Kennedy Family."),
    HumanMessage(content="How many children had Joseph P. Kennedy?"),
]

response = chatModel.invoke(messages)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [ ]:
from langchain_groq import ChatGroq

llamaChatModel = ChatGroq(
    model="groq/compound-mini"
)

In [ ]:
mistralChatModel = ChatGroq(
    model="qwen/qwen3.6-27b"
)

In [ ]:
messages = [
    ("system", "You are an historian expert in the Kennedy family."),
    ("human", "How many members of the family died tragically?"),
]

In [ ]:
llamaResponse = llamaChatModel.invoke(messages)

In [ ]:
print(llamaResponse.content)

In [ ]:
mistralResponse = mistralChatModel.invoke(messages)

In [ ]:
print(mistralResponse.content)

## 02_talk_with_llm_part2.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import OpenAI
llmModel = OpenAI()

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
# This is for completions model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective="curious",
    topic="Tesla"
)

res = llmModel.invoke(llmModelPrompt)
print(res)

In [ ]:
# chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)
print(response.content)


In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input="Name the children and grandchildren of Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [ ]:
print(response.content)

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | chatModel

res = chain.invoke({"input": "How are you?"})
print(res.content)

In [ ]:
from langchain_openai import OpenAI

llmModel = OpenAI()

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser # Fixed import

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an answer key that answers the following question: {question}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [ ]:
json_parser.get_format_instructions()

In [ ]:
res  = json_chain.invoke({"question": "What is the biggest country?"})

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate


In [ ]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [ ]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})

## 03_Data_loader_rag_demo.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
messages = [
    ("system", "You are an helpful assistant."),
    ("human", "tell me about dswithbappy"),
]
response = chatModel.invoke(messages)
print(response.content)

In [ ]:
# Txt data loading
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/be-good.txt")
loaded_data = loader.load()

In [ ]:
# CSV loader

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('./data/Street_Tree_List.csv')

loaded_data = loader.load()
loaded_data

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader

loader = BSHTMLLoader('./data/100-startups.html')
loaded_data = loader.load()

loaded_data

In [ ]:
# Pdf data

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./data/5pages.pdf')

loaded_data = loader.load_and_split()

In [ ]:
import wikipedia
from langchain_community.document_loaders import WikipediaLoader

# 1. Declare a custom user-agent (use your own app name and email description)
wikipedia.set_user_agent("my-langchain-application/1.0 (contact@example.com)")

# 2. Run the loader exactly as before
loader = WikipediaLoader(query="Tesla", load_max_docs=1)
loaded_data = loader.load()[0].page_content

print(loaded_data[:500])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("human", "Answer this {question}, here is some extra {context}"),
    ]
)

messages = chat_template.format_messages(
    name="Tesla",
    question="Tell me about tesla",
    context=loaded_data
)

In [ ]:
response = chatModel.invoke(messages)

## 04_RAG_components.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/be-good.txt")

loaded_data = loader.load()

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
texts = text_splitter.create_documents([loaded_data[0].page_content])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=26,
    chunk_overlap=4
)

text = recursive_splitter.split_text(loaded_data[0].page_content)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

In [ ]:
chunks_of_text =     [
        "Hi there!",
        "Hello!",
        "What's your name?",
        "Bond, James Bond",
        "Hello Bond!"
    ]

In [ ]:
embeddings = embeddings_model.embed_documents(chunks_of_text)

In [ ]:
print(embeddings[0][:5])

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

# Load the document, split it into chunks, embed each chunk and load it into the vector store.
loaded_document = TextLoader('./data/state_of_the_union.txt').load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

chunks_of_text = text_splitter.split_documents(loaded_document)



In [ ]:
vector_db = Chroma.from_documents(chunks_of_text, OpenAIEmbeddings())

In [ ]:
question = "What did the president say about the John Lewis Voting Rights Act?"

response = vector_db.similarity_search(question)

print(response[0].page_content)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/state_of_the_union.txt")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loaded_document = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

chunks_of_text = text_splitter.split_documents(loaded_document)

embeddings = OpenAIEmbeddings()

vector_db = FAISS.from_documents(chunks_of_text, embeddings)

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 1})

In [ ]:
response = retriever.invoke("what did he say about ketanji brown jackson?")
response

## 05_Basic_RAG_Implementation.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = TextLoader("./data/state_of_the_union.txt")
loaded_document = loader.load()

In [ ]:
loaded_document

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks_of_text = text_splitter.split_documents(loaded_document)

In [ ]:
embeddings = OpenAIEmbeddings()

vector_db = FAISS.from_documents(chunks_of_text, embeddings)

In [ ]:
retriever = vector_db.as_retriever()

In [ ]:
response = retriever.invoke("what did he say about ketanji brown jackson?")

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

In [ ]:
response = retriever.invoke("what did he say about ketanji brown jackson?")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI()


In [ ]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])


chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)


In [ ]:
response = chain.invoke("what did he say about ketanji brown jackson?")

In [ ]:
type(response)

## 06_LCEL.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a curious fact about {politician}")

chain = prompt | chatModel | StrOutputParser()

In [ ]:
chain.invoke({"politician": "JFK"})

In [ ]:
from langchain.chains import LLMChain

prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

traditional_chain = LLMChain(
    llm=chatModel,
    prompt=prompt
)

traditional_chain.predict(soccer_player="Maradona")

In [ ]:
chain = prompt | chatModel | StrOutputParser()

chain.invoke({"soccer_player": "Ronaldo"})

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

chain = prompt | chatModel | output_parser

chain.invoke({"soccer_player": "Ronaldo"})

In [ ]:
for s in chain.stream({"soccer_player": "Ronaldo"}):
    print(s, end="", flush=True)

In [ ]:
chain.batch([{"soccer_player": "Ronaldo"}, {"soccer_player": "Messi"}])

## 07_LCEL_builtin_runnables.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

In [ ]:
chain.invoke("Bappy")

In [ ]:
def russian_lastname(name: str) -> str:
    return f"{name}ovich"

In [ ]:
from langchain_core.runnables import RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_lastname)

In [ ]:
chain.invoke("Bappy")

In [ ]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b": RunnableLambda(russian_lastname)
    }
)

In [ ]:
chain.invoke("Bappy")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [ ]:
def russian_lastname_from_dictionary(person):
    return person["name"] + "ovich"

In [ ]:
chain = RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "soccer_player": RunnableLambda(russian_lastname_from_dictionary),
        "operation_c": RunnablePassthrough(),
    }
) | prompt | model | output_parser

In [ ]:
chain.invoke({
    "name1": "Jordam",
    "name": "Abram"
})

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-3.5-turbo")

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is dswithbappy?")

In [ ]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-3.5-turbo")

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question": "What is dswithbappy?", "language": "Pirate English"})

## 08_LCEL_little_more.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [ ]:
chain = prompt | model | output_parser

chain.invoke({"soccer_player": "Ronaldo"})

In [ ]:
chain = prompt | model.bind(stop=["Ronaldo"]) | output_parser
chain.invoke({"soccer_player": "Ronaldo"})

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a sentence about {politician}")

chain = prompt | model | StrOutputParser()

In [ ]:
chain.invoke("Chamberlain")

In [ ]:
historian_prompt = ChatPromptTemplate.from_template("Was {politician} positive for Humanity?")

composed_chain = {"politician": chain} | historian_prompt | model | StrOutputParser()

In [ ]:
composed_chain.invoke({"politician": "Lincoln"})

In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)

model = ChatOpenAI()

chain1 = prompt1 | model | StrOutputParser()

chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"politician": "Miterrand", "language": "Telugu"})

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
import bs4
from langsmith import Client as LangSmithClient  # Use native LangSmith client
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

# Pull the standard RAG prompt securely
client = LangSmithClient()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("What is Task Decomposition?")

## 09_Memory_langchain.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. Define the Prompt Template (Unchanged, but modernized imports)
prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template(
            "You are a nice chatbot having a conversation with a human."
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

# 2. Build the Core LCEL Chain
# (Assuming 'llm' is already initialized, e.g., ChatOpenAI, ChatAnthropic, etc.)
base_chain = prompt | llm

# 3. Manage Session Memories Globally
# In modern LangChain, an dictionary stores history for different users/sessions.
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 4. Wrap the base chain with message history tracking
conversation = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)


In [ ]:
# First interaction
response1 = conversation.invoke(
    {"question": "Hi! My name is Vinay."},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response1.content)

# Second interaction (It will remember your name!)
response2 = conversation.invoke(
    {"question": "What is my name?"},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response2.content)


In [ ]:

response3 = conversation.invoke(
    {"question": "I have moved 33 times."},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response3.content)

In [ ]:

response4 = conversation.invoke(
    {"question": "If my average moving distance was 100 miles, how many miles took all my moves?"},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response4.content)

In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. Define the base conversation prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, context-aware chatbot assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 2. Replicate a strict 'k=3' window using trim_messages
# Setting token_counter=len treats every message entry as exactly 1 unit.
# max_tokens=6 ensures we only retain the latest 6 messages (3 human + 3 AI turns).
trimmer = trim_messages(
    max_tokens=6,
    strategy="last",
    token_counter=len,
    start_on="human",       # Ensures the trimmed window never accidentally cuts off an AI response first
    include_system=True     # Prevents the trimmer from accidentally deleting your System prompt instruction
)

# 3. Build an executable LCEL pipeline with inline trimming
# The prompt history field is filtered dynamically right before hitting the LLM model
chain = (
    {
        "history": lambda x: trimmer.invoke(x["history"]),
        "input": lambda x: x["input"]
    }
    | prompt 
    | llm
)

# 4. Bind the pipeline to normal session state tracking
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation_window = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


In [ ]:
config = {"configurable": {"session_id": "chat_session_abc"}}

# Turn 1
print(conversation_window.invoke({"input": "My favorite color is Blue."}, config).content)
# Turn 2
print(conversation_window.invoke({"input": "I work as a software engineer."}, config).content)
# Turn 3
print(conversation_window.invoke({"input": "I love eating pizza."}, config).content)
# Turn 4
print(conversation_window.invoke({"input": "I have a cat named Whiskers."}, config).content)

# Turn 5: Test the memory window limits (k=3)!
# It will remember the cat and the pizza, but it will have forgotten the color Blue (Turn 1).
response = conversation_window.invoke({"input": "What is my favorite color?"}, config)
print(response.content) 


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# Initialize the modern in-memory history tracker
history = InMemoryChatMessageHistory()

# Add messages exactly as before
history.add_user_message("hi!")
history.add_ai_message("whats up?")


## 11_langgraph_demo.ipynb

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=3)
search.invoke("Who are the top stars of the 2024 Eurocup?")

In [ ]:
tools = [search]

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)

In [ ]:
from langchain_core.messages import HumanMessage

response = agent_executor.invoke({"messages": [HumanMessage(content="Where is the soccer Eurocup 2024 played?")]})

response["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

response = agent_executor.invoke({"messages": [HumanMessage(content="What is Python?")]})

response["messages"]

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="When and where will it be the 2024 Eurocup final match?")]}
):
    print(chunk)
    print("----")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "001"}}

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="Who won the 2024 soccer Eurocup?")]}, config
):
    print(chunk)
    print("----")

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="Who were the top stars of that winner team?")]}, config
):
    print(chunk)
    print("----")

In [ ]:
config = {"configurable": {"thread_id": "001"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="About what soccer team we were talking?")]}, config
):
    print(chunk)
    print("----")

In [ ]:
config = {"configurable": {"thread_id": "002"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="About what soccer team we were talking?")]}, config
):
    print(chunk)
    print("----")

## 12_langsmith_demo.ipynb

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [ ]:
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")

In [ ]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Please resposne to the user request only based on the given context"),
        ("user","Question:{question}\nContext:{context}")


    ]
)

In [ ]:
model=ChatOpenAI(model="gpt-3.5-turbo") 
 
output_parser=StrOutputParser()

In [ ]:
chain= prompt| model | output_parser

In [ ]:
question="can you summarize the story?"



context=""" 

A dragon is a magical animal, an animal that is not real. However, we find stories of dragons in all cultures around the world. Dragons can breathe fire and fly, and they like to eat people. Dragons like to collect treasure, gold and jewels, and hide it in their home. In the TV series Game of Thrones, Daenerys had several dragons.

When you tell the future, you say what is going to happen in the future. People tell the future in many ways. Some people look at the lines on your hand to tell your future. Some people use tarot cards to tell the future. Personally, I don’t believe that anyone can tell the future.

"""


In [ ]:
print(chain.invoke({"question":question,"context":context}))